## 1. Setup & Imports

In [1]:
!pip install -q ucimlrepo


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
import torch
import torch.nn as nn
from torch.autograd import Variable
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from numpy.linalg import svd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

Using device: cuda
GPU: NVIDIA RTX A6000
CUDA Version: 12.8


## 2. Load & Prepare the Dataset

In [4]:
print("Fetching dataset from UCI Repository...")
spambase = fetch_ucirepo(id=94)
X_raw = spambase.data.features
y_raw = spambase.data.targets

df = pd.concat([X_raw, y_raw], axis=1)
target_col = y_raw.columns[0]
feature_names = X_raw.columns.tolist()

print(f"Dataset Loaded: {df.shape[0]} rows, {len(feature_names)} features.")
print(f"Feature names: {feature_names}")

Fetching dataset from UCI Repository...
Dataset Loaded: 4601 rows, 57 features.
Feature names: ['word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will', 'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free', 'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit', 'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money', 'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650', 'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology', 'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct', 'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project', 'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference', 'c

In [5]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

scaler = MinMaxScaler()
df[feature_names] = scaler.fit_transform(df[feature_names])

weights = abs(df.corr()[target_col]).drop(target_col).values
weights = weights / (np.linalg.norm(weights) + 1e-8)

bounds = [df[feature_names].min().values, df[feature_names].max().values]

print(f"Dataset Size: {len(df)} samples.")
print(f"Class distribution: {df[target_col].value_counts().to_dict()}")

Dataset Size: 4601 samples.
Class distribution: {0: 2788, 1: 1813}


## 3. Model & Attack

In [6]:
class SpambaseNet(nn.Module):
    def __init__(self, D_in):
        super(SpambaseNet, self).__init__()
        self.layer = nn.Sequential(
            nn.Linear(D_in, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2), nn.Softmax(dim=-1)
        )
    
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            return self.layer(x).squeeze(0)
        return self.layer(x)

In [7]:
def clip(current, low_bound, up_bound, device):
    low_bound = torch.FloatTensor(low_bound).to(device)
    up_bound = torch.FloatTensor(up_bound).to(device)
    return torch.max(torch.min(current, up_bound), low_bound)


def lowProFool_gpu(x, model, weights, bounds, maxiters, alpha, lambda_, device):
    """GPU version of LowProFool. Returns (orig_pred, adv_pred, x_adv as numpy)."""
    x = x.to(device)
    v = torch.FloatTensor(np.array(weights)).to(device)
    
    r = torch.FloatTensor(1e-4 * np.ones(x.shape)).to(device)
    r.requires_grad = True
    
    with torch.no_grad():
        output = model(x)
        orig_pred = output.argmax().cpu().item()
    
    target_pred = 1 - orig_pred
    target = torch.tensor([0., 1.] if target_pred == 1 else [1., 0.]).to(device)
    
    bce = nn.BCELoss()
    
    for _ in range(maxiters):
        if r.grad is not None:
            r.grad.zero_()
        
        output = model(x + r)
        
        loss_bce = bce(output, target)
        loss_l2 = torch.sqrt(torch.sum((v * r) ** 2))
        loss = loss_bce + lambda_ * loss_l2
        
        loss.backward(retain_graph=True)
        
        with torch.no_grad():
            r_new = r - alpha * r.grad
        r = r_new.clone().detach().requires_grad_(True)
    
    x_adv = clip(x + r, bounds[0], bounds[1], device)
    
    with torch.no_grad():
        adv_pred = model(x_adv).argmax().cpu().item()
    
    return orig_pred, adv_pred, x_adv.detach().cpu().numpy()

## 4. Training

In [8]:
X_tensor = torch.FloatTensor(df[feature_names].values)
y_tensor = torch.nn.functional.one_hot(torch.LongTensor(df[target_col].values.astype(int))).float()

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

X_train = X_train.to(device)
X_test = X_test.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

Training set: 3680 samples
Test set: 921 samples


In [9]:
model = SpambaseNet(len(feature_names)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

print("Training model...")
model.train()
for epoch in range(100):
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/100, Loss: {loss.item():.4f}")

model.eval()
print("Training Complete.")

with torch.no_grad():
    train_acc = (model(X_train).argmax(dim=1) == y_train.argmax(dim=1)).float().mean()
    test_acc = (model(X_test).argmax(dim=1) == y_test.argmax(dim=1)).float().mean()
print(f"Train Accuracy: {train_acc:.2%}")
print(f"Test Accuracy: {test_acc:.2%}")

Training model...
Epoch 20/100, Loss: 0.6617
Epoch 40/100, Loss: 0.6105
Epoch 60/100, Loss: 0.5185
Epoch 80/100, Loss: 0.4063
Epoch 100/100, Loss: 0.3148
Training Complete.
Train Accuracy: 89.32%
Test Accuracy: 90.34%


## 5. Adversarial Examples

Running LowProFool with two λ values:
- λ = 0.1 (aggressive) — bigger perturbations, should flip almost everything
- λ = 50 (stealthy) — tries to stay under the radar

In [10]:
ALPHA = 0.1
MAXITERS = 300
LAMBDAS = {'aggressive': 0.1, 'stealthy': 50.0}
N_SAMPLES = min(2000, X_test.shape[0])

print(f"Experiment Parameters:")
print(f"  Alpha: {ALPHA}")
print(f"  Max iterations: {MAXITERS}")
print(f"  Lambda (aggressive): {LAMBDAS['aggressive']}")
print(f"  Lambda (stealthy): {LAMBDAS['stealthy']}")
print(f"  Number of samples: {N_SAMPLES}")

Experiment Parameters:
  Alpha: 0.1
  Max iterations: 300
  Lambda (aggressive): 0.1
  Lambda (stealthy): 50.0
  Number of samples: 921


In [11]:
test_samples = X_test[:N_SAMPLES].cpu()
original_data = test_samples.numpy()

adversarial_data = {}
attack_success = {}

for attack_type, lambda_val in LAMBDAS.items():
    print(f"\n--- Generating {attack_type} adversarial examples (λ={lambda_val}) ---")
    
    adv_samples = []
    successful = 0
    
    for i, x in enumerate(tqdm(test_samples, desc=f"LowProFool ({attack_type})")):
        orig_pred, adv_pred, x_adv = lowProFool_gpu(
            x, model, weights, bounds, MAXITERS, ALPHA, lambda_val, device
        )
        adv_samples.append(x_adv)
        if orig_pred != adv_pred:
            successful += 1
    
    adversarial_data[attack_type] = np.array(adv_samples)
    attack_success[attack_type] = successful / N_SAMPLES
    
    print(f"Success rate ({attack_type}): {attack_success[attack_type]:.2%}")
    
    perturbation = adversarial_data[attack_type] - original_data
    avg_l2_norm = np.mean(np.linalg.norm(perturbation, axis=1))
    print(f"Average L2 perturbation ({attack_type}): {avg_l2_norm:.6f}")


--- Generating aggressive adversarial examples (λ=0.1) ---


LowProFool (aggressive): 100%|██████████| 921/921 [05:41<00:00,  2.70it/s]


Success rate (aggressive): 100.00%
Average L2 perturbation (aggressive): 1.007674

--- Generating stealthy adversarial examples (λ=50.0) ---


LowProFool (stealthy): 100%|██████████| 921/921 [05:13<00:00,  2.94it/s]

Success rate (stealthy): 98.59%
Average L2 perturbation (stealthy): 0.883275


## 6. Random Noise Baseline

Need a control: random Gaussian noise matched in L2 norm to the aggressive attack, so we can tell if the SVD changes are specific to adversarial structure or just a magnitude thing.

In [12]:
aggressive_perturbation = adversarial_data['aggressive'] - original_data
target_l2_norms = np.linalg.norm(aggressive_perturbation, axis=1)

np.random.seed(42)
random_noise = np.random.randn(*original_data.shape)

noise_norms = np.linalg.norm(random_noise, axis=1, keepdims=True)
random_noise = random_noise / (noise_norms + 1e-8) * target_l2_norms.reshape(-1, 1)

noisy_data = np.clip(original_data + random_noise, bounds[0], bounds[1])

print(f"Random noise control created.")
print(f"Average L2 noise magnitude: {np.mean(np.linalg.norm(noisy_data - original_data, axis=1)):.6f}")

Random noise control created.
Average L2 noise magnitude: 0.721835


In [14]:
print(adversarial_data['aggressive'].shape)
print(noisy_data.shape)
print(adversarial_data['stealthy'].shape)

(921, 57)
(921, 57)
(921, 57)


## 7. Iterative Noise Scoring (SVD)

Run the iterative SVD noise scoring on three scenarios:
1) aggressive adversarial samples,
2) stealthy adversarial samples,
3) random noise baseline (`noisy_data`) with matched perturbation magnitude.

In [ ]:
import time

SCORING_N = original_data.shape[0]
TOPK = None  # None => show all removal steps
CHUNK_SIZE = 256


def _to_2d_float32(arr):
    x = np.asarray(arr)
    if x.ndim == 3 and x.shape[1] == 1:
        x = x[:, 0, :]
    elif x.ndim > 2:
        x = x.reshape(x.shape[0], -1)
    return x.astype(np.float32)


def compute_iterative_scores_torch(X_np, chunk_size=256, device=None, log_every=50):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    num_instances = X_np.shape[0]
    index_removed_at_step = np.zeros(num_instances, dtype=int)
    noise_score_at_step = np.zeros(num_instances, dtype=np.float64)

    X_running = torch.tensor(X_np, dtype=torch.float32, device=device)
    singular_values_shape = None
    t_all_start = time.perf_counter()

    for i in range(num_instances):
        t_loop_start = time.perf_counter()

        s_running = torch.linalg.svdvals(X_running)
        if singular_values_shape is None:
            singular_values_shape = tuple(s_running.shape)

        row_nonzero_mask = torch.any(X_running != 0, dim=1)
        candidate_indices = torch.nonzero(row_nonzero_mask, as_tuple=False).squeeze(1)

        if candidate_indices.numel() == 0:
            break

        best_score = -float('inf')
        best_idx = int(candidate_indices[0].item())
        comp_count = 0

        for start in range(0, candidate_indices.numel(), chunk_size):
            chunk = candidate_indices[start:start + chunk_size]
            batch_size = int(chunk.shape[0])

            X_batch = X_running.unsqueeze(0).repeat(batch_size, 1, 1)
            X_batch[torch.arange(batch_size, device=device), chunk, :] = 0

            s_new_batch = torch.linalg.svdvals(X_batch)
            min_len = min(s_running.shape[-1], s_new_batch.shape[-1])
            diff = s_new_batch[:, :min_len] - s_running[:min_len].unsqueeze(0)
            scores = torch.linalg.norm(diff, dim=1)

            chunk_best_score, chunk_best_pos = torch.max(scores, dim=0)
            chunk_best_score_val = float(chunk_best_score.item())
            if chunk_best_score_val > best_score:
                best_score = chunk_best_score_val
                best_idx = int(chunk[int(chunk_best_pos.item())].item())

            comp_count += batch_size

        index_removed_at_step[i] = best_idx
        noise_score_at_step[i] = best_score
        X_running[best_idx, :] = 0

        if (i + 1) % log_every == 0 or i < 3 or i == num_instances - 1:
            t_loop = time.perf_counter() - t_loop_start
            print(
                f"  Processed {i + 1}/{num_instances} | "
                f"loop_time={t_loop:.3f}s | computations={comp_count}"
            )

    total_time = time.perf_counter() - t_all_start
    print(f"Total scoring time: {total_time:.3f}s")

    return noise_score_at_step, index_removed_at_step, singular_values_shape


scenario_inputs = {
    'aggressive': _to_2d_float32(adversarial_data['aggressive'][:SCORING_N]),
    'stealthy': _to_2d_float32(adversarial_data['stealthy'][:SCORING_N]),
}

if 'noisy_data' in globals():
    scenario_inputs['random_noise_baseline'] = _to_2d_float32(noisy_data[:SCORING_N])
else:
    raise ValueError('Need noisy_data in memory to run the random noise baseline.')

svd_results = {}
scoring_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using torch device for SVD scoring: {scoring_device}")
print(f"Scenarios: {list(scenario_inputs.keys())}")

for name, X_scenario in scenario_inputs.items():
    print(f"\n=== Iterative SVD noise scoring: {name} | shape={X_scenario.shape} ===")

    noise_scores, index_removed_at_step, singular_values_shape = compute_iterative_scores_torch(
        X_scenario,
        chunk_size=CHUNK_SIZE,
        device=scoring_device,
        log_every=50,
    )

    topk = len(noise_scores) if TOPK is None else min(TOPK, len(noise_scores))
    top_pairs = [
        (int(index_removed_at_step[i]), float(noise_scores[i]))
        for i in range(topk)
    ]

    svd_results[name] = {
        'backend': f'torch/{scoring_device}',
        'shape': X_scenario.shape,
        'singular_values_shape': singular_values_shape,
        'top_removed': top_pairs,
        'noise_scores': noise_scores,
        'index_removed_at_step': index_removed_at_step,
    }

    print(f"Reporting {topk} removed instances for {name}:")
    for rank, (idx, score) in enumerate(top_pairs, start=1):
        print(f"  {rank:4d}. Original Index: {idx:4d}, Noise Score: {score:.6f}")

print("\nDone. Results are in svd_results for aggressive, stealthy, and random_noise_baseline.")

Using torch device for SVD scoring: cuda
Scenarios: ['aggressive', 'stealthy', 'random_noise_baseline']

=== Iterative SVD noise scoring: aggressive | shape=(921, 57) ===
  Processed 1/921 | loop_time=8.986s | computations=921
  Processed 2/921 | loop_time=8.418s | computations=920
  Processed 3/921 | loop_time=8.431s | computations=919
